In [1]:
import pandas as pd
import numpy as np
import os
import pyotp
import datetime as dt
from datetime import datetime
import json
import time
import requests

from kiteconnect import KiteConnect
from kiteconnect import KiteTicker
from dotenv import load_dotenv
load_dotenv()  # Load credentials from .env file (not committed to version control)
creds = {
    'user_id': os.getenv('ZERODHA_USER_ID'),
    'password': os.getenv('ZERODHA_PASSWORD'),
    'totp_key': os.getenv('ZERODHA_TOTP_KEY'),
    'api_key': os.getenv('ZERODHA_API_KEY'),
    'api_secret': os.getenv('ZERODHA_API_SECRET')
}
base_url = "https://kite.zerodha.com"
login_url = "https://kite.zerodha.com/api/login"
twofa_url = "https://kite.zerodha.com/api/twofa"
instruments_url = "https://api.kite.trade/instruments"

session = requests.Session()
response = session.post(login_url,data={'user_id':creds['user_id'],'password':creds['password']})
request_id = json.loads(response.text)['data']['request_id']
twofa_pin = pyotp.TOTP(creds['totp_key']).now()
# twofa_pin =149166
response_1 = session.post(twofa_url,data={'user_id':creds['user_id'],'request_id':request_id,'twofa_value':twofa_pin,'twofa_type':'totp'})
kite = KiteConnect(api_key=creds['api_key'])
kite_url = kite.login_url()
try:
    session.get(kite_url)
except Exception as e:
    e_msg = str(e)
#print(e_msg)
request_token = e_msg.split('request_token=')[1].split(' ')[0].split('&action')[0]
print('Successful Login with Request Token:{}'.format(request_token))
data = kite.generate_session(request_token,creds['api_secret'])
access_token =  data['access_token']

kite.set_access_token(access_token)
#Get last traded prices for Nifty and BankNifty
# Initialize Kite Ticker
kws = KiteTicker(creds['api_key'], access_token)


Successful Login with Request Token:3kCx3FE4KGWX9NujJs6wnLjUz13SQ8PX


In [2]:
pip install pandas


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
print(os.environ['SPARK_LOCAL_IP'])

127.0.0.1


In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .config("spark.driver.extraJavaOptions", "-XX:+UseG1GC") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .config("spark.eventLog.gcMetrics.youngGenerationGarbageCollectors", "G1 Concurrent GC") \
    .config("spark.eventLog.gcMetrics.oldGenerationGarbageCollectors", "G1 Concurrent GC") \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 17:37:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 59616)
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/Library/Frameworks/Pyt

In [5]:
def round_to_hundred(num):
    if num % 100 >= 50:
        return num + (100 - num % 100)
    else:
        return num - (num % 100)
    
def round_to_fifty(num):
    if num % 50 >= 25:
        return num + (50 - num % 50)
    else:
        return num - (num % 50)
    

def get_ATM_Strike(instrument_token, ronding_number):
    _ltp = (kite.ltp(instrument_token)[str(instrument_token)]["last_price"])
    #get ATM Strikes
    if ronding_number == 100:
        return round_to_hundred(_ltp)
    if ronding_number == 50:
        return round_to_fifty(_ltp)
    

In [6]:
# importing module 
import pyspark 

from datetime import datetime, date, time, timedelta

# importing sparksession from 
# pyspark.sql module 
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType, TimestampType, LongType, FloatType
from datetime import datetime
from pyspark.sql.functions import desc, asc, like, lit, abs, col, to_date, round
import pyspark.sql.functions as f



# creating sparksession and giving 
# an app name 
spark = SparkSession.builder.appName('sparkdf').getOrCreate() 

# Define the schema for the DataFrame
schema = StructType([
    StructField("instrument_token", IntegerType(), nullable=False),
    StructField("exchange_token", StringType(), nullable=False),
    StructField("tradingsymbol", StringType(), nullable=False),
    StructField("name", StringType(), nullable=False),
    StructField("last_price", DoubleType(), nullable=False),
    StructField("expiry", DateType(), nullable=False),
    StructField("strike", DoubleType(), nullable=False),
    StructField("tick_size", DoubleType(), nullable=False),
    StructField("lot_size", IntegerType(), nullable=False),
    StructField("instrument_type", StringType(), nullable=False),
    StructField("segment", StringType(), nullable=False),
    StructField("exchange", StringType(), nullable=False)
])

# Read Parquet files into a DataFrame
Banknifty_Options_DF_From_File = spark.read.parquet("DataFiles/Instruments/Banknifty_Options", schema=schema)
Banknifty_Futures_DF_From_File = spark.read.parquet("DataFiles/Instruments/Banknifty_Futures", schema=schema)
Nifty_Options_DF_From_File = spark.read.parquet("DataFiles/Instruments/Nifty_Options", schema=schema)
Nifty_Futures_DF_From_File = spark.read.parquet("DataFiles/Instruments/Nifty_Futures", schema=schema)

# Define the schema for the expiry DataFrame
schema = StructType([
    StructField("name", StringType(), nullable=False),
    StructField("current_week_expiry", DateType(), nullable=False),
    StructField("current_month_expiry", DateType(), nullable=False),
    StructField("next_week_expiry", DateType(), nullable=False),
    StructField("next_month_expiry", DateType(), nullable=False)
])
Nifty_Banknifty_Expiries_DF = spark.read.parquet("DataFiles/Instruments/Nifty_Banknifty_Expiries", schema=schema)

26/03/10 17:37:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [7]:
Nifty_Banknifty_Expiries_DF.show()

+---------+-------------------+--------------------+----------------+-----------------+
|     name|current_week_expiry|current_month_expiry|next_week_expiry|next_month_expiry|
+---------+-------------------+--------------------+----------------+-----------------+
|BANKNIFTY|         2026-03-30|          2026-03-30|      2026-04-28|       2026-04-28|
|    NIFTY|         2026-03-10|          2026-03-30|      2026-03-17|       2026-04-28|
+---------+-------------------+--------------------+----------------+-----------------+



In [8]:

Banknifty_current_week_expiry   = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "BANKNIFTY").select("current_week_expiry").collect()[0][0]
Banknifty_current_month_expiry  = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "BANKNIFTY").select("current_month_expiry").collect()[0][0]
Banknifty_next_week_expiry      = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "BANKNIFTY").select("next_week_expiry").collect()[0][0]
Banknifty_next_month_expiry     = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "BANKNIFTY").select("next_month_expiry").collect()[0][0]


Nifty_current_week_expiry       = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "NIFTY").select("current_week_expiry").collect()[0][0]
Nifty_current_month_expiry      = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "NIFTY").select("current_month_expiry").collect()[0][0]
Nifty_next_week_expiry          = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "NIFTY").select("next_week_expiry").collect()[0][0]
Nifty_next_month_expiry         = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "NIFTY").select("next_month_expiry").collect()[0][0]

print(
Banknifty_current_week_expiry   ,
Banknifty_current_month_expiry  ,
Banknifty_next_week_expiry      ,
Banknifty_next_month_expiry     )
print(
Nifty_current_week_expiry       ,
Nifty_current_month_expiry      ,
Nifty_next_week_expiry          ,
Nifty_next_month_expiry         )

26/03/10 17:37:44 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Young Generation), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/03/10 17:37:44 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Old Generation, G1 Young Generation), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


2026-03-30 2026-03-30 2026-04-28 2026-04-28
2026-03-10 2026-03-30 2026-03-17 2026-04-28


In [9]:

#define functions to calculate column
def calculate_Strike_level_type(col1):
    if col1 < 0:
        return 'ITM'
    elif col1 > 0:
        return 'OTM'
    else:
        return 'ATM'
    
def calculate_Strike_level_Name(col1):
    if col1 < 0:
        return 'ATM' + "" + str.replace(str(col1),".0","")
    elif col1 > 0:
        return 'ATM' + "+" + str.replace(str(col1),".0","")
    else:
        return 'ATM'
    
    
def get_Options_DF(Options_DF_From_File, ATM_Strikes, current_week_expiry, strike_range, strikes):
#Select ATM and 9 ITM and 9 OTM strikes
    Options_DF = Options_DF_From_File.filter(Options_DF_From_File['expiry'] == current_week_expiry)\
    .filter(Options_DF_From_File["strike"] <= ATM_Strikes + (strike_range*strikes))\
    .filter(Options_DF_From_File["strike"] >= ATM_Strikes - (strike_range*strikes)).orderBy(asc("strike"))

    #add customer columns
    Options_DF = Options_DF.withColumn("ATM_Strike", lit(ATM_Strikes))
    Options_DF = Options_DF.withColumn("Strike_level", ((Options_DF["strike"]-lit(ATM_Strikes))/strike_range))

    #register functions as UDF with return type
    udfcalculate_Strike_level_type = f.udf(calculate_Strike_level_type, StringType())
    udfcalculate_Strike_level_Name = f.udf(calculate_Strike_level_Name, StringType())

    #Add customer column
    Options_DF = Options_DF.withColumn("Strike_level_type", udfcalculate_Strike_level_type(Options_DF["Strike_level"]))
    Options_DF = Options_DF.withColumn("Strike_level_Name", udfcalculate_Strike_level_Name(Options_DF["Strike_level"]))

    #Select only required columns
    Options_DF = Options_DF.select( "name", "instrument_token","expiry", "Strike_level_Name", "strike", "Strike_level_type", "lot_size", "instrument_type", "segment")
    return  Options_DF


In [10]:
#Get all trading instruments
# instrument=kite.instruments()

#Get Instrument tokens for Nifty and BankNifty
Banknifty_instrument_token = 260105# [instrument for instrument in instrument if instrument['name'] == 'NIFTY BANK'][0]['instrument_token']
Nifty_instrument_token = 12014082#[instrument for instrument in instrument if instrument['name'] == 'NIFTY'][0]['instrument_token']
#get ATM Strikes
Banknifty_ATM_Strikes = get_ATM_Strike(Banknifty_instrument_token,100)
Nifty_ATM_Strikes     = get_ATM_Strike(Nifty_instrument_token,50)
print("Nifty_ATM_Strikes = ", Nifty_ATM_Strikes, "Banknifty_ATM_Strikes = " , Banknifty_ATM_Strikes)


Nifty_ATM_Strikes =  0 Banknifty_ATM_Strikes =  57000.0


In [11]:

BankNifty_Use_Custom_Strike=True
BankNifty_Custom_Strike=49300
Nifty_Use_Custom_Strike=True
Nifty_Custom_Strike=22950

#For Banknifty Current Week Expiry
if BankNifty_Use_Custom_Strike:
    Banknifty_Options_DF = get_Options_DF(Banknifty_Options_DF_From_File, BankNifty_Custom_Strike ,Banknifty_current_week_expiry,100,9)
else:
    Banknifty_Options_DF = get_Options_DF(Banknifty_Options_DF_From_File, get_ATM_Strike(Banknifty_instrument_token,100) ,Banknifty_current_week_expiry,100,9)
#For Nifty Current Week Expiry
if Nifty_Use_Custom_Strike:
    Nifty_Options_DF = get_Options_DF(Nifty_Options_DF_From_File, Nifty_Custom_Strike, Nifty_current_week_expiry, 50,9)
else:
    Nifty_Options_DF = get_Options_DF(Nifty_Options_DF_From_File, get_ATM_Strike(Nifty_instrument_token,50), Nifty_current_week_expiry, 50,9)


In [12]:
def get_historical_data(token,from_date, to_date, interval):
    historical_data = kite.historical_data(token,from_date,to_date,interval,oi=True)
    # print(historical_data)
    # Define the schema for the DataFrame
    schema = StructType([
        StructField("date", TimestampType(), nullable=True),
        StructField("open", StringType(), nullable=True),
        StructField("high", StringType(), nullable=True),
        StructField("low", StringType(), nullable=True),
        StructField("close", StringType(), nullable=True),
        StructField("volume", StringType(), nullable=True),
        StructField("oi", StringType(), nullable=True),
        StructField("day", DateType(), nullable=True)
    ])
    df = spark.createDataFrame(historical_data, schema=schema)
    df = df.orderBy(asc("date")).withColumn('day',col('date').cast("date"))
    return df

In [13]:
# token = 14503682
# to_date = datetime.now()
# from_date = datetime.now()-timedelta(days=7)
# interval = "minute"

# get_historical_data(token,from_date, to_date, interval).show()


In [14]:
#Write Banknifty current week expiry

#create list of istuments for the data pull
Banknifty_instrument_list_ = Banknifty_Options_DF.select('instrument_token').orderBy(asc("instrument_token")).collect()
#create list of istuments for the data pull
Nifty_instrument_list_ = Nifty_Options_DF.select('instrument_token').orderBy(asc("instrument_token")).collect()

#Set variables
to_date = datetime.now()
from_date = datetime.now()-timedelta(days=7)
interval = "3minute"


In [15]:

#run loop to write data
for i in range(0,len(Banknifty_instrument_list_)):
    get_historical_data(Banknifty_instrument_list_[i]["instrument_token"],from_date, to_date, interval).coalesce(1).write.format('Delta').mode("overwrite").partitionBy("day").parquet("DataFiles/HistoricalData/"+interval+"/"+str(Banknifty_current_week_expiry)+"/"+str(Banknifty_instrument_list_[i]["instrument_token"]))
    get_historical_data(Nifty_instrument_list_[i]["instrument_token"],from_date, to_date, interval).coalesce(1).write.format('Delta').mode("overwrite").partitionBy("day").parquet("DataFiles/HistoricalData/"+interval+"/"+str(Nifty_current_week_expiry)+"/"+str(Nifty_instrument_list_[i]["instrument_token"]))



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py:154: DeprecationWarning: This process (pid=20363) is multi-threaded, use of fork() may lead to deadlocks in the child.


In [16]:
# # Banknifty_Options_DF.show()

# def get_Strike_Token(Options_DF,Name, Expiry, Strike_level_Name, CE_OR_PE):
#     return Options_DF\
#     .filter(Options_DF['name'] == Name)\
#     .filter(Options_DF['expiry'] == Expiry)\
#     .filter(Options_DF['Strike_level_Name'] == Strike_level_Name)\
#     .filter(Options_DF['instrument_type'] == CE_OR_PE).select(("instrument_token")).collect()[0]["instrument_token"]


# def get_Strike_OHLC_data(Options_DF,Name, Expiry, Strike_level_Name, CE_OR_PE, interval):
#     Options_DF=  Options_DF\
#     .filter(Options_DF['name'] == Name)\
#     .filter(Options_DF['expiry'] == Expiry)\
#     .filter(Options_DF['Strike_level_Name'] == Strike_level_Name)\
#     .filter(Options_DF['instrument_type'] == CE_OR_PE)
#     instrument_token =  Options_DF.collect()[0]["instrument_token"]
#     strike =  Options_DF.collect()[0]["strike"]
#     schema = StructType([
#         StructField("date", TimestampType(), nullable=True),
#         StructField("open", StringType(), nullable=True),
#         StructField("high", StringType(), nullable=True),
#         StructField("low", StringType(), nullable=True),
#         StructField("close", StringType(), nullable=True),
#         StructField("volume", StringType(), nullable=True),
#         StructField("oi", StringType(), nullable=True),
#         StructField("day", DateType(), nullable=True)
#     ])
#     return spark.read.parquet("DataFiles/HistoricalData/"+interval+"/"+str(Banknifty_current_week_expiry)+"/"+str(instrument_token), schema=schema)\
#         .filter(col("volume") > 0)\
#         .withColumn("instrument_token",lit(instrument_token))\
#         .withColumn("name",lit(Name))\
#         .withColumn("strike",lit(strike))\
#         .withColumn("expiry",lit(Expiry))\
#         .withColumn("ohlc4",round((col('open')+col('high')+col('low')+col('close'))/4,2))

In [17]:
# Banknifty_Options_DF.show()


In [18]:
# #Banknifty ATM Strike data
# #now get data for the token
# get_Strike_OHLC_data(
#                 Options_DF          = Banknifty_Options_DF,
#                 Name                = "BANKNIFTY",
#                 Expiry              = Banknifty_current_week_expiry,
#                 Strike_level_Name   = 'ATM-1',
#                 CE_OR_PE            = 'CE',
#                 interval            = interval).orderBy(desc("date")).show()

26/03/10 18:24:33 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 577983 ms exceeds timeout 120000 ms
26/03/10 18:24:33 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/10 18:24:37 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o